# Explaining Messy Health Data - Solutions

The **2022 Behavioral Risk Factor Surveillance System (BRFSS)** contains 445,132 survey responses and 328 columns. The table is much larger and messier than any one analysis requires.

The goal is to turn coded survey responses into understandable evidence. You will decode categories, inspect distributions, fit regression models, summarize correlated measurements with PCA, and test why accuracy can fail on an uncommon health outcome.

BRFSS is a telephone survey. Most values are self-reported, optional questions are not asked everywhere, and codes such as `77`, `88`, and `99` can represent special responses rather than quantities. The models below are demonstrations, not clinical tools.

## 1. Load the complete survey

The first run downloads an approximately 81 MB official archive and extracts an approximately 1.1 GB XPT file. Loading all 328 columns can take several seconds and temporarily requires several gigabytes of memory.

This is a useful starting point: the complete dataset makes it clear that choosing and interpreting variables is part of data analysis.

Source: [CDC BRFSS 2022 annual data](https://www.cdc.gov/brfss/annual_data/annual_2022.html).

In [ ]:
from pathlib import Path
import os
import shutil
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)

In [ ]:
DATA_URL = "https://www.cdc.gov/brfss/annual_data/2022/files/LLCP2022XPT.zip"
data_path = Path(os.environ.get("BRFSS_XPT_PATH", "LLCP2022.XPT"))

if not data_path.exists():
    archive_path = Path("LLCP2022XPT.zip")
    print("Downloading the official BRFSS archive...")
    urllib.request.urlretrieve(DATA_URL, archive_path)

    with zipfile.ZipFile(archive_path) as archive:
        member = archive.namelist()[0]
        with archive.open(member) as source, data_path.open("wb") as destination:
            shutil.copyfileobj(source, destination)
    archive_path.unlink()

raw = pd.read_sas(data_path, format="xport")
print(f"Loaded {raw.shape[0]:,} rows and {raw.shape[1]:,} columns")
print(f"DataFrame memory: {raw.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

`raw` is the untouched source table. Keep it available so every transformation can be checked against the original codes. pandas Copy-on-Write allows a second DataFrame to share unchanged data with `raw` until a column is actually transformed.

In [ ]:
raw.head()

The preview contains abbreviated names and unexplained numbers. Search the column names to find candidates, then confirm each definition in the BRFSS questionnaire and codebook. A plausible name is not enough.

In [ ]:
search_terms = (
    "BIRTHSEX", "MENT", "PHYS", "SLEP", "SMOK", "DRNK", "BMI", "HTIN"
)
possible_columns = [
    column for column in raw.columns
    if any(term in column for term in search_terms)
]
possible_columns

## 2. Decode the fields before analysing them

Create one full-width `survey` DataFrame. It retains all source fields, but the fields used below receive readable names and documented values.

Categorical codes become text. Special codes in numerical questions become zero only when the code explicitly means “none”; “don't know,” refused, and unavailable responses become missing numerical values. `raw` is never modified.

In [ ]:
sex_labels = {1: "Male", 2: "Female", 7: "Don't know", 9: "Refused"}
general_health_labels = {
    1: "Excellent", 2: "Very good", 3: "Good", 4: "Fair",
    5: "Poor", 7: "Don't know", 9: "Refused",
}
yes_no_labels = {1: "Yes", 2: "No", 7: "Don't know", 9: "Refused"}


def clean_unhealthy_days(series):
    """Decode 88 as zero days and remove non-numeric response codes."""
    return series.replace({88: 0, 77: np.nan, 99: np.nan}).where(
        lambda values: values.between(0, 30)
    )


no_alcohol = raw["ALCDAY4"].eq(888)
average_drinks = (
    raw["AVEDRNK3"].where(raw["AVEDRNK3"].between(1, 76))
    .mask(no_alcohol | raw["AVEDRNK3"].eq(88), 0)
)
binge_days = (
    raw["DRNK3GE5"].where(raw["DRNK3GE5"].between(1, 76))
    .mask(no_alcohol | raw["DRNK3GE5"].eq(88), 0)
)
maximum_drinks = (
    raw["MAXDRNKS"].where(raw["MAXDRNKS"].between(1, 76))
    .mask(no_alcohol | raw["MAXDRNKS"].eq(88), 0)
)
smoking_frequency_score = np.select(
    [
        raw["SMOKDAY2"].eq(1),
        raw["SMOKDAY2"].eq(2),
        raw["SMOKDAY2"].eq(3) | raw["SMOKE100"].eq(2),
    ],
    [2.0, 1.0, 0.0],
    default=np.nan,
)
smoking_status = np.select(
    [
        raw["SMOKDAY2"].eq(1),
        raw["SMOKDAY2"].eq(2),
        raw["SMOKDAY2"].eq(3),
        raw["SMOKE100"].eq(2),
    ],
    ["Every day", "Some days", "Former smoker", "Never smoked 100 cigarettes"],
    default="Unknown / unavailable",
)

refusal_codes = {
    "GENHLTH": 9, "PHYSHLTH": 99, "MENTHLTH": 99, "POORHLTH": 99,
    "SLEPTIM1": 99, "SMOKE100": 9, "SMOKDAY2": 9, "ALCDAY4": 999,
    "AVEDRNK3": 99, "DRNK3GE5": 99, "MAXDRNKS": 99,
    "BIRTHSEX": 9,
}
refusal_flags = pd.DataFrame({
    column: raw[column].eq(code)
    for column, code in refusal_codes.items()
})

# Rename and decode only documented fields. The other BRFSS columns remain available.
survey = (
    raw.rename(columns={
        "BIRTHSEX": "birth_sex",
        "GENHLTH": "general_health",
        "PHYSHLTH": "poor_physical_days",
        "MENTHLTH": "poor_mental_days",
        "SLEPTIM1": "sleep_hours",
        "SMOKDAY2": "smoking_status",
        "AVEDRNK3": "average_drinks",
        "DRNK3GE5": "binge_days",
        "MAXDRNKS": "maximum_drinks",
        "HTIN4": "height_inches",
        "_BMI5": "bmi",
        "_AGE80": "age",
        "ADDEPEV3": "depression_diagnosis",
    })
    .assign(
        birth_sex=raw["BIRTHSEX"].map(sex_labels).fillna("Not asked"),
        general_health=raw["GENHLTH"].map(general_health_labels),
        poor_physical_days=clean_unhealthy_days(raw["PHYSHLTH"]),
        poor_mental_days=clean_unhealthy_days(raw["MENTHLTH"]),
        sleep_hours=raw["SLEPTIM1"].where(raw["SLEPTIM1"].between(1, 24)),
        smoking_status=smoking_status,
        smoking_frequency_score=smoking_frequency_score,
        average_drinks=average_drinks,
        binge_days=binge_days,
        maximum_drinks=maximum_drinks,
        height_inches=raw["HTIN4"],
        bmi=raw["_BMI5"] / 100,
        age=raw["_AGE80"],
        depression_diagnosis=raw["ADDEPEV3"].map(yes_no_labels),
        refused_questions=refusal_flags.sum(axis=1),
        unavailable_questions=raw[list(refusal_codes)].isna().sum(axis=1),
    )
)

print(f"raw: {raw.shape[1]} columns; survey: {survey.shape[1]} columns")
print("raw codes are still unchanged:", raw.loc[0, ["MENTHLTH", "SLEPTIM1"]].to_dict())

Now `describe(include="all")` can summarize numerical measurements and readable categories together. Numerical rows show means and percentiles; categorical rows show counts, unique values, and the most frequent category.

In [ ]:
interesting_columns = [
    "birth_sex", "general_health", "poor_physical_days", "poor_mental_days",
    "sleep_hours", "smoking_status", "average_drinks", "binge_days",
    "maximum_drinks", "height_inches", "bmi", "age",
    "depression_diagnosis", "refused_questions",
]

survey[interesting_columns].describe(include="all").T

## 3. Height and birth-sex responses

`birth_sex` came from an optional module, so most respondents were not asked the question. The available responses include `Male`, `Female`, `Don't know`, and `Refused`.

`Don't know` is a meaningful answer rather than a bad row. For example, a binary question may not describe every person with intersex traits, but this dataset does **not** identify why a particular respondent chose `Don't know`. `Not asked` describes survey design, and `Refused` provides too little information for biological interpretation.

In [ ]:
height_columns = ["birth_sex", "height_inches", "age"]
display(survey[height_columns].head())

birth_sex_counts = survey["birth_sex"].value_counts(dropna=False)
birth_sex_counts

The counts show two separate limitations. `Not asked` dominates because the question was optional, while `Don't know` is a small but genuine response category. Neither should be silently changed to `Male` or `Female`.

In [ ]:
height_categories = ["Female", "Male", "Don't know"]
height_plot = survey.loc[
    survey["birth_sex"].isin(height_categories)
].dropna(subset=["height_inches"])

height_summary = height_plot.groupby("birth_sex")["height_inches"].agg(
    respondents="size", mean="mean", standard_deviation="std"
).reindex(height_categories)
display(height_summary.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label, color in [("Female", "#d95f8d"), ("Male", "#277da1")]:
    values = height_plot.loc[height_plot["birth_sex"] == label, "height_inches"]
    axes[0].hist(
        values, bins=np.arange(48, 85), density=True,
        alpha=0.5, label=label, color=color,
    )
axes[0].set(
    title="Height distributions overlap",
    xlabel="Self-reported height (inches)", ylabel="Density",
)
axes[0].legend(title="Birth-sex response")

boxplot_values = [
    height_plot.loc[height_plot["birth_sex"] == label, "height_inches"]
    for label in height_categories
]
axes[1].boxplot(boxplot_values, tick_labels=height_categories, vert=False)
axes[1].set(
    title="Keep the Don't know responses visible",
    xlabel="Self-reported height (inches)", ylabel="Birth-sex response",
)
plt.tight_layout()
plt.show()

The `Male` and `Female` means differ, but their distributions overlap. The much smaller `Don't know` group can be described, but its height distribution cannot explain why respondents chose that answer.

Standard deviation measures spread around a group mean; it does not prove that an unusual value is an error or reveal a person's sex traits.

In [ ]:
height_model_data = height_plot[height_plot["birth_sex"].isin(["Female", "Male"])]

# scikit-learn expects a 2D numerical array: rows by features.
X_height = height_model_data[["height_inches"]].to_numpy()
y_height = height_model_data["birth_sex"].eq("Male").to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X_height, y_height, test_size=0.25, random_state=42, stratify=y_height
)
height_model = LogisticRegression().fit(X_train, y_train)

height_grid = np.linspace(50, 84, 300)
male_probability = height_model.predict_proba(height_grid.reshape(-1, 1))[:, 1]
test_accuracy = height_model.score(X_test, y_test)
ambiguous = (male_probability >= 0.40) & (male_probability <= 0.60)

print(f"Height-only test accuracy: {test_accuracy:.1%}")
print(
    "40%-60% probability region:",
    f"{height_grid[ambiguous].min():.1f} to {height_grid[ambiguous].max():.1f} inches",
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(height_grid, male_probability, color="#553c9a", linewidth=3)
ax.axhspan(0.40, 0.60, color="#f6bd60", alpha=0.35, label="Model should admit uncertainty")
ax.set(
    title="A height-only model produces probabilities, not biological facts",
    xlabel="Height (inches)",
    ylabel="Predicted probability of birth-sex response = Male",
    ylim=(0, 1),
)
ax.legend()
plt.show()

The model uses only respondents who answered `Male` or `Female`, because logistic regression requires a defined binary target. `Don't know` remains in `survey` and in the descriptive analysis; it is not treated as an error or forced into either class.

The labelled pandas column becomes a two-dimensional NumPy array because scikit-learn expects rows by features. `np.linspace()` creates the height values used for the probability curve.

This is a probability about a survey response, not a biological rule. Height cannot determine birth sex or identify intersex traits.

## 4. Physical health, mental health, and sleep

The cleaning step already decoded `88` as zero unhealthy days and removed non-numeric response codes. The resulting columns can now be summarized and modeled without accidentally treating `88` as eighty-eight days in a thirty-day question.

In [ ]:
wellbeing_columns = ["poor_physical_days", "poor_mental_days", "sleep_hours"]
survey[wellbeing_columns].describe().round(2)

In [ ]:
regression_rows = survey.dropna(
    subset=["poor_physical_days", "poor_mental_days"]
)
physical_to_mental = LinearRegression().fit(
    regression_rows[["poor_physical_days"]], regression_rows["poor_mental_days"]
)

x_line = np.linspace(0, 30, 100)
y_line = physical_to_mental.predict(pd.DataFrame({"poor_physical_days": x_line}))
plot_rows = regression_rows.sample(5_000, random_state=42)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    plot_rows["poor_physical_days"], plot_rows["poor_mental_days"],
    alpha=0.08, s=15, color="#277da1"
)
ax.plot(x_line, y_line, color="#d62828", linewidth=3, label="Linear regression")
ax.set(
    title=f"Physical and mental unhealthy days (R² = {physical_to_mental.score(regression_rows[["poor_physical_days"]], regression_rows["poor_mental_days"]):.2f})",
    xlabel="Physically unhealthy days in past 30",
    ylabel="Mentally unhealthy days in past 30",
)
ax.legend()
plt.show()

The line summarizes an average association. It does not show that physically unhealthy days cause mentally unhealthy days. The responses are also bounded between 0 and 30, so a straight line is only a rough summary.

In [ ]:
sleep_rows = survey.dropna(subset=["sleep_hours", "poor_mental_days"])
sleep_model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    LinearRegression(),
).fit(sleep_rows[["sleep_hours"]], sleep_rows["poor_mental_days"])

sleep_grid = pd.DataFrame({"sleep_hours": np.linspace(3, 12, 200)})
sleep_means = sleep_rows.groupby("sleep_hours")["poor_mental_days"].agg(["mean", "size"])
sleep_means = sleep_means.loc[(sleep_means.index >= 3) & (sleep_means.index <= 12)]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    sleep_means.index, sleep_means["mean"],
    s=np.sqrt(sleep_means["size"]) * 2, color="#277da1", label="Observed mean"
)
ax.plot(
    sleep_grid["sleep_hours"], sleep_model.predict(sleep_grid),
    color="#d62828", linewidth=3, label="Quadratic regression"
)
ax.set(
    title="A curved model can represent a non-linear sleep pattern",
    xlabel="Self-reported sleep (hours)",
    ylabel="Average mentally unhealthy days",
)
ax.legend()
plt.show()

The curved model can represent patterns that a straight line misses. It still describes an association rather than a cause, and unusual sleep responses deserve inspection before interpretation.

## 5. Refused and unavailable questions

The early cleaning step counted explicit refusals using the documented code for each selected question. Missing values were counted separately because they often mean a question was skipped or not asked.

The refusal count might be worth investigating, but it is not a psychological measurement. Privacy, language, survey routing, and interviewer effects are alternative explanations.

In [ ]:
refusal_counts = survey["refused_questions"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(refusal_counts.index.astype(str), refusal_counts.values, color="#6a4c93")
ax.set(
    title="Explicit refusals across 13 selected questions",
    xlabel="Questions explicitly refused",
    ylabel="Respondents",
)
plt.show()

refusal_counts

## 6. PCA: visualize covariance between measurements

Principal component analysis places correlated measurements along shared directions. First standardize the numerical fields so a variable does not dominate merely because it uses larger units.

The variable correlation map then places each original measurement according to its correlation with PC1 and PC2:

- arrows pointing in similar directions indicate positive covariance;
- arrows pointing in opposite directions indicate negative covariance;
- arrows near right angles indicate a weak relationship in these two components;
- longer arrows are represented better by the two-dimensional map.

In [ ]:
pca_features = [
    "poor_physical_days", "poor_mental_days", "sleep_hours",
    "smoking_frequency_score", "average_drinks", "binge_days", "maximum_drinks",
    "bmi", "age", "height_inches",
]
pca_rows = survey.dropna(subset=pca_features + ["general_health"])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_rows[pca_features])

print(f"Complete rows used by PCA: {len(pca_rows):,} of {len(survey):,}")
print(type(X_scaled), X_scaled.shape)
print("Largest absolute standardized mean:", np.abs(X_scaled.mean(axis=0)).max().round(10))
print("Standard deviations:", X_scaled.std(axis=0).round(2))

In [ ]:
pca = PCA().fit(X_scaled)
scores = pca.transform(X_scaled)

# Correlations place each original variable in the first two-component coordinate system.
variable_coordinates = pd.DataFrame(
    pca.components_[:2].T * np.sqrt(pca.explained_variance_[:2]),
    index=pca_features,
    columns=["PC1", "PC2"],
)
feature_groups = {
    "poor_physical_days": "Wellbeing", "poor_mental_days": "Wellbeing",
    "sleep_hours": "Wellbeing", "bmi": "Wellbeing",
    "smoking_frequency_score": "Substance use", "average_drinks": "Substance use",
    "binge_days": "Substance use", "maximum_drinks": "Substance use",
    "age": "Demographics", "height_inches": "Demographics",
}
group_colors = {
    "Wellbeing": "#d1495b", "Substance use": "#277da1", "Demographics": "#2a9d8f"
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].plot(
    range(1, len(pca_features) + 1), pca.explained_variance_ratio_,
    marker="o", color="#553c9a",
)
axes[0].set(
    title="Variance explained by each component",
    xlabel="Principal component", ylabel="Share of variance explained",
    xticks=range(1, len(pca_features) + 1),
)

for feature, coordinates in variable_coordinates.iterrows():
    group = feature_groups[feature]
    axes[1].arrow(
        0, 0, coordinates["PC1"], coordinates["PC2"],
        color=group_colors[group], alpha=0.8, head_width=0.025, length_includes_head=True,
    )
    axes[1].text(
        coordinates["PC1"] * 1.08, coordinates["PC2"] * 1.08,
        feature.replace("_", " "), color=group_colors[group], fontsize=9,
        ha="center", va="center",
    )

circle = plt.Circle((0, 0), 1, fill=False, color="gray", linestyle="--", alpha=0.5)
axes[1].add_patch(circle)
axes[1].axhline(0, color="gray", linewidth=0.7)
axes[1].axvline(0, color="gray", linewidth=0.7)
axes[1].set(
    title="Variable correlation map",
    xlabel="Correlation with PC1", ylabel="Correlation with PC2",
    xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), aspect="equal",
)
for group, color in group_colors.items():
    axes[1].scatter([], [], color=color, label=group)
axes[1].legend(loc="lower left")
plt.tight_layout()
plt.show()

variable_coordinates.round(2)

The first direction is dominated by alcohol-use measurements: average drinks, binge-drinking days, and maximum drinks point together. The second direction is dominated by mentally and physically unhealthy days, while sleep points partly in the opposite direction.

This makes the covariance visible, but a component is not a diagnosis and does not establish causation.

In [ ]:
score_frame = pd.DataFrame(
    scores[:, :2], columns=["PC1", "PC2"], index=pca_rows.index
).assign(general_health=pca_rows["general_health"])

health_order = ["Excellent", "Very good", "Good", "Fair", "Poor"]
health_colors = {
    "Excellent": "#0072B2",
    "Very good": "#56B4E9",
    "Good": "#009E73",
    "Fair": "#E69F00",
    "Poor": "#D55E00",
}
plot_sample = score_frame.sample(
    min(8_000, len(score_frame)), random_state=42
)

fig, ax = plt.subplots(figsize=(9, 6))
for category in health_order:
    group = plot_sample[plot_sample["general_health"] == category]
    ax.scatter(
        group["PC1"],
        group["PC2"],
        color=health_colors[category],
        alpha=0.35,
        s=14,
        edgecolors="none",
        label=f"{category} (n={len(group):,})",
        rasterized=True,
    )

ax.set(
    title="Individual responses overlap in the PCA map",
    xlabel=f"PC1 ({pca.explained_variance_ratio_[0]:.1%} of variance)",
    ylabel=f"PC2 ({pca.explained_variance_ratio_[1]:.1%} of variance)",
)
ax.legend(title="General health", markerscale=2, frameon=False)
fig.tight_layout()
plt.show()

Each point is one sampled respondent, colored by their general-health response. The transparency reveals dense regions and makes the substantial overlap between categories visible; PCA does not divide respondents into distinct groups.

PCA used complete rows only. Respondents with unavailable answers are absent from this map. Imputation could retain them, but it would add assumptions rather than recover known answers.

## 7. Why accuracy can be dangerous

Define an uncommon outcome: reporting **14 or more mentally unhealthy days** in the past month. This is not a diagnosis.

Compare a classifier that always says “no” with ordinary logistic regression and a recall-focused version. **Recall** asks: of the respondents who actually reported frequent distress, what fraction did the model flag?

In [ ]:
risk_rows = survey.dropna(subset=["poor_mental_days"]).assign(
    frequent_mental_distress=lambda frame: frame["poor_mental_days"].ge(14)
)
risk_features = [
    "poor_physical_days", "sleep_hours", "smoking_frequency_score",
    "average_drinks", "bmi", "age",
]
X_risk = risk_rows[risk_features]
y_risk = risk_rows["frequent_mental_distress"]

print(y_risk.value_counts())
print(f"Positive-class prevalence: {y_risk.mean():.1%}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_risk, y_risk, test_size=0.25, random_state=42, stratify=y_risk
)

models = {
    "Always says no": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": make_pipeline(
        SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2_000)
    ),
    "Recall-focused logistic": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=2_000, class_weight="balanced"),
    ),
}

results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    predictions[name] = prediction
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, prediction),
        "precision": precision_score(y_test, prediction, zero_division=0),
        "recall": recall_score(y_test, prediction, zero_division=0),
    })

pd.DataFrame(results).set_index("model").round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, name in zip(axes, ["Always says no", "Recall-focused logistic"]):
    ConfusionMatrixDisplay.from_predictions(
        y_test, predictions[name],
        display_labels=["Not frequent", "Frequent distress"],
        colorbar=False, ax=ax,
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

The “always no” model receives high accuracy because most respondents are in the negative class, but its recall is zero. Balanced logistic regression finds more positive cases at the cost of more false alarms and lower accuracy.

The appropriate trade-off depends on what happens after a flag and on the relative costs of missed cases and false alarms. For a low-cost follow-up screen, recall may matter more than raw accuracy.

## Final investigation

Choose one claim suggested by these results and try to disprove it.

1. Find the relevant readable fields in `survey` and their original codes in `raw`.
2. Check the official definitions and missing-value rules.
3. Plot the distributions before fitting a model.
4. Fit one scikit-learn model and report an appropriate metric.
5. State one association, one alternative explanation, and one limitation.

Possible questions include whether the sleep pattern changes with age, whether refusal rates differ across questions, or whether the distress classifier has equal recall across birth-sex response groups.

## Takeaways

- Loading data is only the beginning; coded values must be interpreted before calculation.
- Keep an unchanged source table so every transformation remains auditable.
- Text categories, missing responses, and real numerical measurements require different treatments.
- `describe(include="all")` gives a useful first view after decoding.
- Regression summarizes relationships but does not establish causes.
- A PCA variable map makes covariance visible through direction and distance.
- Group categories can have different average PCA positions while individuals still overlap.
- High accuracy can hide zero recall on the cases that matter.